In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import cv2
import pandas as pd
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from PIL import Image

DATA_ROOT = '/content/drive/MyDrive/dp/data/my/data'

subdirectories = [
    d for d in os.listdir(DATA_ROOT)
    if os.path.isdir(os.path.join(DATA_ROOT, d))
]

image_counts = {}
image_extensions = ['.jpg', '.jpeg', '.png']

for subdirectory in subdirectories:
    subdirectory_path = os.path.join(DATA_ROOT, subdirectory)
    count = 0

    for filename in os.listdir(subdirectory_path):
        filepath = os.path.join(subdirectory_path, filename)

        if os.path.isfile(filepath):
            _, file_extension = os.path.splitext(filename)

            if file_extension.lower() in image_extensions:
                count += 1

    image_counts[subdirectory] = count

print(image_counts)

!pip install opencv-python scikit-image

def calculate_snr(image):
    mean_squared_error = np.mean(image**2)

    if mean_squared_error == 0:
        return 0

    signal_power = mean_squared_error
    noise_power = np.var(image)

    if noise_power == 0:
        return float('inf')

    return 10 * np.log10(signal_power / noise_power)

def calculate_psnr(reference_image, distorted_image):
    return peak_signal_noise_ratio(reference_image, distorted_image)

def calculate_ssim(reference_image, distorted_image):
    min_height = min(reference_image.shape[0], distorted_image.shape[0])
    min_width = min(reference_image.shape[1], distorted_image.shape[1])

    reference_image_resized = cv2.resize(reference_image, (min_width, min_height))
    distorted_image_resized = cv2.resize(distorted_image, (min_width, min_height))

    if len(reference_image_resized.shape) == 3:
        reference_image_resized = cv2.cvtColor(reference_image_resized, cv2.COLOR_BGR2GRAY)

    if len(distorted_image_resized.shape) == 3:
        distorted_image_resized = cv2.cvtColor(distorted_image_resized, cv2.COLOR_BGR2GRAY)

    return structural_similarity(reference_image_resized, distorted_image_resized)

def is_corrupted(image_path):
    try:
        img = Image.open(image_path)
        img.verify()
        return False
    except Exception:
        return True

def is_blurred(image, threshold=100.0):
    if len(image.shape) == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    fm = cv2.Laplacian(image, cv2.CV_64F).var()

    return fm < threshold

image_quality_results = []

for subdirectory in subdirectories:
    subdirectory_path = os.path.join(DATA_ROOT, subdirectory)

    for filename in os.listdir(subdirectory_path):
        image_path = os.path.join(subdirectory_path, filename)

        if os.path.isfile(image_path):
            _, file_extension = os.path.splitext(filename)

            if file_extension.lower() in image_extensions:
                is_img_corrupted = is_corrupted(image_path)

                result = {
                    'class': subdirectory,
                    'filename': filename,
                    'is_corrupted': is_img_corrupted,
                    'resolution_height': None,
                    'resolution_width': None,
                    'bit_depth': None,
                    'aspect_ratio': None,
                    'snr': None,
                    'psnr': None,
                    'ssim': None,
                    'is_blurred': None
                }

                if not is_img_corrupted:
                    try:
                        img = cv2.imread(image_path)

                        if img is not None:
                            height, width = img.shape[:2]

                            result['resolution_height'] = height
                            result['resolution_width'] = width

                            if height > 0:
                                result['aspect_ratio'] = width / height

                            result['snr'] = calculate_snr(img)
                            result['is_blurred'] = is_blurred(img)

                        else:
                            result['is_corrupted'] = True

                    except Exception as e:
                        print(f"Error processing image {image_path}: {e}")
                        result['is_corrupted'] = True

                image_quality_results.append(result)

print(f"Processed {len(image_quality_results)} images.")

image_quality_df = pd.DataFrame(image_quality_results)

aggregated_results = image_quality_df.groupby('class').agg(
    total_images=('filename', 'count'),
    corrupted_images=('is_corrupted', lambda x: x.sum()),
    blurred_images=('is_blurred', lambda x: x.sum()),
    resolution_height_mean=('resolution_height', 'mean'),
    resolution_height_std=('resolution_height', 'std'),
    resolution_width_mean=('resolution_width', 'mean'),
    resolution_width_std=('resolution_width', 'std'),
    aspect_ratio_mean=('aspect_ratio', 'mean'),
    aspect_ratio_std=('aspect_ratio', 'std'),
    snr_mean=('snr', 'mean'),
    snr_std=('snr', 'std')
).reset_index()

display(aggregated_results)